# Quantification du MLP Fashion-MNIST pour la Nexys

Ce notebook quantifie le modèle `784 -> 64 -> 32 -> 10` entraîné dans `01b_train_fashion.ipynb`.

- Les échelles sont calibrées sur les 60 000 images d'entraînement.
- La simulation bit-exacte tourne sur les 10 000 images de test.
- La référence est le modèle float32 : on compare les classes prédites.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'src' / 'py' / 'fpga_targets.py').is_file())
source_dir = PROJECT_ROOT / 'src' / 'py'
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from quantization import TARGET_PROFILES, run_quantization, validate_quantization_config

## Configuration

`generate_rtl` écrit dans `src/rtl/generated/src_nexys_sequential` les sources à ajouter au projet Vivado (mode `sequential` uniquement pour l'instant).

In [ ]:
config = {
    'name': 'fashion',
    'model_path': 'data/fashion/mlp_fashion.pt',
    'dataset_path': 'data/fashion/dataset_fashion.npz',
    'output_root': 'data/quantized_data',

    'target': 'nexys',
    'architecture': 'sequential',    # sequential or pipeline

    'weight_bits': 16,
    'bias_bits': 16,
    'act_bits': 16,
    'activation_negative_slope': 1 / 64,

    'test_samples': 100,
    'simulation_samples': None,      # None = toutes les images de test
    'fail_on_overflow': True,
    'generate_report': True,
    'generate_rtl': True,
}

config = validate_quantization_config(config)
TARGET_PROFILES[config['target']]

## Exécution complète

In [ ]:
run = run_quantization(config, project_root=PROJECT_ROOT)

## Résultats de la simulation

In [ ]:
pd.Series(run.simulation_summary, name='value').to_frame()

In [ ]:
run.simulation_report

## Images où la carte et le float32 ne prédisent pas la même classe

In [ ]:
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

quantized = np.load(run.dataset_path)
different = np.nonzero(quantized['class_float'] != quantized['class_board'])[0]

pd.DataFrame({
    'image': different,
    'vraie classe': [classes[quantized['y'][i]] for i in different],
    'float32': [classes[quantized['class_float'][i]] for i in different],
    'carte': [classes[quantized['class_board'][i]] for i in different],
})

## Fichiers générés

In [ ]:
print(f'Dossier : {run.paths.output_dir}')
print(f'Manifeste : {run.manifest_path}')
print(f'Dataset quantifié : {run.dataset_path}')
print(f"Sources Vivado : {run.rtl_project['generated_rtl_dir']}")